# Notebook 4 — Analyse principale : exposition air x mutations

**Objectif :** Construire la table d'analyse finale (cohorte + air + radon + socio-eco), calculer les metriques d'exposition par fenetres temporelles, et realiser les regressions logistiques et de Poisson.

---

## Donnees en entree

- `../data/1_patients/patients_geocoded_clean_idf_2018_2023.csv` — cohorte geocodee
- `../data/1_patients/patients_radon_score_final.csv` — score radon
- `../data/3_analyses/pneumodetect_cohorte_idf_ineris_data.csv` — series temporelles air + temperature
- `../data/0_brut/socio_eco/edi2021-iris-fm.xlsx` — indice de defavorisation INSEE
- `../data/0_brut/geospatial/icpe/icpe.geojson/icpe_idf.shp` — sites ICPE Seveso
- `../data/0_brut/geospatial/tmja/` — trafic routier TMJA 2024

## Donnees en sortie

- `../data/2_exposition/patients_exposition_metrique_after_2018.csv` — metriques d'exposition
- `../data/2_exposition/patients_exposition_metrique_after_2018_2023.csv` — metriques 2018-2023
- `../data/3_analyses/patients_clinics_air_metrics.csv` — table d'analyse complete
- `../data/3_analyses/patients_clinics_air_metrics_groupes.csv` — table avec groupes mutation
- `../data/1_patients/patients_geocoded_clean_idf_2018_2023_groups.csv` — cohorte avec groupes
- `../figures/statistiques/` — forest plots, heatmaps, correlations

---

> **RGPD** : Ce notebook traite des donnees de sante pseudonymisees.
> Les fichiers `data/4_confidentiel/` ne doivent jamais etre versionnés sur git.
> **Chemin racine :** `h:/PFE Loice/Notebooks/Loice_Canc-air_2025/loice_pneumodetect/`

# LUNG-CANC'AIR — Notebook principal

> **Séquence d'exécution :**
```
0   → Charger data + df_clinique
1   → CFG (fenetre_ans, seuils arbitraires — polluants_retenus défini en 5b)
2   → Construire df_final (toutes variables)
3   → Exploration visuelle
4   → Seuils BH — 1 seul seuil retenu par polluant
5   → Heatmaps + FAMD
5b  → ★ Décision polluants_retenus (après heatmaps)
5c  → Analyses monovariées (variables finales)
6   → Tableaux de caractéristiques
7   → ★ Régressions principales (variables BH) ★
7b  → Lasso sélection exposition (covariables cliniques forcées)
8   → Mutations individuelles
9   → Sensibilité (sexe, sous-groupes)
10  → Dose-réponse
11  → Variables % du temps
11b → Sensibilité multi-fenêtres × polluants
11e → ★ C* sur données brutes
11f → PM25_pct_supC* + P*
11g → Seuils Youden cumul et pente
11c → Seuils Youden variables df_final
11h → ★ Synthèse comparative manuelle vs auto
11i → ★ Régressions finales comparatives
12  → Synthèse générale
```


---
## PARTIE 0 — Chemins des fichiers

In [ ]:
# =============================================================================
# CHEMINS 
# =============================================================================

CHEMINS = {
    # Données de pollution journalière (INERIS)
    'pollution' : r"../data/3_analyses/pneumodetect_cohorte_idf_ineris_data.csv",

    # Données cliniques (patients geocodés)
    'clinique'  : r"../data/1_patients/patients_geocoded_clean_idf_2018_2023.csv",

    # Indice de défaveur sociale (EDI2021)
    'edi'       : r"../data/0_brut/socio_eco/edi2021-iris-fm.xlsx",

    # Shapefile routes nationales (TMJA)
    'tmja'      : r"../data/0_brut/geospatial/tmja/",

    # Fichier ICPE (shapefile)
    'icpe'      : r"../data/0_brut/geospatial/icpe/icpe.geojson/icpe_idf.shp",

    # Score radon géologique continu (0-100) — déjà joint sur pseudo_provisoire
    'radon'     : r"../data/1_patients/patients_radon_score_final.csv",
}

print("✅ Chemins définis")


---
## PARTIE 1 — ★ CONFIGURATION ★

> `polluants_retenus` n'est PAS défini ici — il sera défini en **Partie 5b**
> après avoir vu les heatmaps de corrélation.


In [ ]:
# =============================================================================
# ★ CONFIGURATION CENTRALE ★
# Modifier ici : fenetre_ans, seuils arbitraires, ICPE, filtres
# polluants_retenus → défini en Partie 5b après heatmaps
# =============================================================================

CFG = {

    # ── Fenêtre temporelle ────────────────────────────────────────────────────
    'fenetre_ans'      : 3,             # ← CHANGER ICI (3 | 5 | 10 | 15)

    # ── Polluants — Calcul (tous calculés) ───────────────────────────────────
    'polluants_calcul' : ['PM25', 'PM10', 'NO2', 'O3'],

    # ── polluants_retenus → sera défini en Partie 5b après heatmaps ──────────
    # (laisser vide ici)
    'polluants_retenus': [],

    # ── Seuils arbitraires % du temps (testés en Partie 4) ───────────────────
    'seuils_pct' : {
        'PM25': [5, 10, 15, 25, 35],
        # 'PM10': [35, 45, 50, 60, 80, 90],
        'O3'  : [100, 120, 180],
    },

    # ── Paramètres routiers ───────────────────────────────────────────────────
    'shapefile_tmja' : CHEMINS['tmja'],
    'buffer_rn_m'    : 500,

    # ── Paramètres ICPE ───────────────────────────────────────────────────────
    'icpe_path'     : CHEMINS['icpe'],
    'rayons_icpe_m' : [3000],            # ← changer (ex: [3000, 5000])
    'icpe_types'    : {'NS':'NS', 'SB':'SB', 'SH':'SH'},

    # ── Variables modèles — remplies automatiquement ─────────────────────────
    'cumul'    : [],   # Partie 2
    'tendance' : [],   # Partie 5b
    'pct_pm25' : [],   # Partie 4
    'pct_pm10' : [],   # Partie 4
    'pct_o3'   : [],   # Partie 4
    'icpe'     : [],   # Partie 2

    # ── Variables cliniques ───────────────────────────────────────────────────
    'inclure_age'    : True,
    'inclure_paquet' : True,
    'inclure_sexe'   : True,
    'inclure_edi'    : True,
    'inclure_trafic' : True,
   

    'inclure_radon'  : False,   # ← True = inclure le score radon géologique continu (0-100) dans tous les modèles
    'radon_var'      : 'radon_score', 
    'geo_notation'   : 'NOTATION',    
    # Note : cohorte Bassin parisien → radon attendu comme confondant mineur (pas de granites)
    # L'analyse de sensibilité (Partie 9b) permet de vérifier et d'en discuter dans les limitations

    # IPG — Indice de Pollution Global
    # C'est une note composite de tous les polluants standardisés (z-score moyen PM25+PM10+NO2+O3)
    # Avantage : résout la multicolinéarité entre polluants, 1 seule variable d'exposition globale
    # Pour l'activer : ajouter 'IPG' à polluants_retenus ET mettre inclure_ipg à True
    # Exemple : 'polluants_retenus': ['IPG']  ou  ['PM25', 'O3', 'IPG']
    'inclure_ipg'    : True,    # ← True = IPG inclus dans tous les modèles

    # ── Filtres population ────────────────────────────────────────────────────
    'filtre_sexe'       : None,
    'filtre_fumeur'     : None,
    'filtre_histologie' : None,
    'filtre_custom'     : {},

    # ── Dose-réponse ──────────────────────────────────────────────────────────
    'polluant_dr_AB'  : None,
    'polluant_dr_CD'  : None,
    'n_quintiles'     : 5,
    'col_distance'    : 'dist_ICPE_SH_m',
    'tranches_km'     : [0, 1, 3, 5, 10, 999],
}

print("✅ Configuration définie")
print(f"   Fenêtre      : {CFG['fenetre_ans']} an(s)")
print(f"   Polluants calcul : {CFG['polluants_calcul']}")
print(f"   polluants_retenus → à définir en Partie 5b après heatmaps")


---
## PARTIE 2 — Construction de df_final

> Exécuter une seule fois.

In [ ]:
import sys
sys.path.append('src')
import importlib
import lungcancair_analyses as lca
importlib.reload(lca)


In [ ]:

data, df_clinique = lca.charger_donnees(CHEMINS)
df_air = lca.calculer_variables_air(data, df_clinique, CFG)
df_air = lca.ajouter_socioeco(df_air, df_clinique, CHEMINS['edi'])
df_air, gdf_patients = lca.ajouter_routier(df_air, df_clinique, CFG)
df_air = lca.ajouter_icpe(df_air, gdf_patients, CFG)
df_air   = lca.ajouter_radon(df_air, df_clinique, CHEMINS['radon'], CFG)
df_final = lca.construire_df_final(data, df_clinique, df_air)

# Colonnes cumul (selon fenetre_ans, tous polluants calculés)
# polluants_retenus pas encore défini → cumul complet pour exploration
CFG['cumul'] = lca.detecter_vars_cumul(df_final)   # tous polluants
CFG['icpe']  = lca.detecter_vars_icpe(df_final, CFG)

# Dose-réponse (provisoire — mis à jour en 5b)
CFG['polluant_dr_AB'] = next((c for c in CFG['cumul'] if 'PM25' in c), CFG['cumul'][0] if CFG['cumul'] else None)
CFG['polluant_dr_CD'] = next((c for c in CFG['cumul'] if 'O3'   in c), CFG['cumul'][0] if CFG['cumul'] else None)

print(f"\n✅ df_final : {len(df_final)} patients × {len(df_final.columns)} variables")
print(f"   cumul (tous) : {CFG['cumul']}")
print(f"   icpe         : {CFG['icpe']}")
print(f"   → polluants_retenus à définir en Partie 5b")


In [ ]:
df_final

In [ ]:
df_final.to_csv(r'../data/patients_geocoded_clean_idf_2018_2023_groups.csv', sep=';')

#### 📊 Résultats — Construction df_final

*À compléter après exécution.*

---
## PARTIE 3 — Exploration visuelle

In [ ]:
lca.explorer_distribution_polluants(df_final)

#### 📊 Résultats — Distribution polluants

*À compléter après exécution.*

In [ ]:
# IPG — Aperçu rapide de la distribution
if 'IPG' in df_final.columns:
    print('IPG — Score composite pollution (z-score moyen PM25+PM10+NO2+O3)')
    print(df_final['IPG'].describe().round(3))
    print(f"\nRappel : IPG > 0 = exposition au-dessus de la moyenne de la cohorte")
    print(f"          IPG < 0 = exposition en-dessous de la moyenne")
else:
    print('⚠️  IPG non disponible dans df_final — vérifier calculer_variables_air()')

In [ ]:
lca.explorer_distribution_pct(df_final)

#### 📊 Résultats — Distribution % du temps

*À compléter après exécution.*

In [ ]:
lca.explorer_tendances(df_final)

#### 📊 Résultats — Tendances MM365+MK

*À compléter après exécution.*

--- ## PARTIE 3b — Exploration visuelle du radon

> Visualise la distribution du radon et de la lithologie par groupe.
> Inclut un test Chi² pour chaque comparaison.

In [ ]:
lca.explorer_distribution_radon(df_final)

#### 📊 Résultats — Distribution radon

*À compléter après exécution.*

---
## PARTIE 4 — Seuils critiques + Benjamini-Hochberg

> Teste tous les seuils arbitraires × tous les polluants.
> **1 seul seuil retenu par polluant** = le plus significatif après BH.
> Résultats utilisés pour comparaison en Partie 11h.


In [ ]:
# Utiliser polluants_calcul pour l'exploration des seuils
cfg_4 = {**CFG, 'polluants_retenus': CFG['polluants_calcul']}
df_res_seuils, seuils_retenus = lca.chercher_seuils_critiques(df_final, cfg_4)

# Stocker pour comparaison future (Partie 11h)
CFG['_seuils_bh_tous'] = seuils_retenus   # tous polluants
CFG['_df_res_seuils']  = df_res_seuils

print(f"\n── Seuils retenus (1 par polluant) ──")
for pol, s in seuils_retenus.items():
    print(f"  {pol:<8} : {s}")
print("\n→ Ces seuils seront filtrés sur polluants_retenus en Partie 5b")


#### 📊 Résultats — Seuils critiques + BH

*À compléter après exécution.*

---
## PARTIE 5 — Corrélation & FAMD

> **Objectif :** identifier les corrélations entre polluants → décider `polluants_retenus` en Partie 5b.

In [ ]:
# Heatmap 1 — tous les polluants calculés
vars_h1 = {}
for pol in CFG['polluants_calcul']:
    cols_p = [c for c in df_final.columns
              if c.startswith(f'{pol}_cumul_') and 'pct' not in c and 'manq' not in c]
    cols_moy = [f'{pol}_moyenne'] if f'{pol}_moyenne' in df_final.columns else []
    if cols_p or cols_moy:
        vars_h1[f'{pol}'] = cols_moy + cols_p[:1]
lca.heatmap_correlation(df_final, vars_h1)


#### 📊 Résultats — Heatmap 1 — corrélations inter-polluants

*À compléter après exécution.*

In [ ]:
# FAMD — toutes les mutations disponibles automatiquement
mutations_dispo = sorted([col.replace('mutation_','')
                           for col in df_final.columns
                           if col.startswith('mutation_')])
print(f"Mutations disponibles ({len(mutations_dispo)}) : {mutations_dispo}")

for mut in mutations_dispo:
    df_final[f'mut_{mut}_bin'] = lca.porte_mutation(df_final, mut).map(
        {True:'Positif', False:'Négatif'})

# Variables continues pour FAMD
vars_fc = (
    CFG['cumul']    +
    CFG['tendance'] +
    ['age_diagnostic','paquet_annee','quintileEDI2021','indice_trafic'] +
    CFG['icpe']
)
vars_fc = [v for v in vars_fc if v in df_final.columns]

# Variables catégorielles — groupes + TOUTES les mutations
vars_fcat = (
    ['sexe','histologie_groupe','groupe_AB','groupe_CD','groupe_AC_BD'] +
    [f'mut_{m}_bin' for m in mutations_dispo if f'mut_{m}_bin' in df_final.columns]
)
vars_fcat = [v for v in vars_fcat if v in df_final.columns]

print(f"FAMD : {len(vars_fc)} variables continues + {len(vars_fcat)} catégorielles")
print(f"  Mutations incluses : {mutations_dispo}")

famd, coords, var_exp, df_famd = lca.run_famd(df_final, vars_fc, vars_fcat)


#### 📊 Résultats — FAMD — exploration

*À compléter après exécution.*

---
## PARTIE 5b — ★ Décision : polluants_retenus ★

> **Après avoir vu les heatmaps**, décider quels polluants inclure dans les régressions.
>
> Règles basées sur les corrélations observées :
> - Si r(PM25, PM10) > 0.90 → garder UN SEUL (PM25 recommandé car plus fine)
> - Si r(NO2, PM25)  > 0.70 → garder UN SEUL (PM25 recommandé)
> - PM25 et O3 sont généralement indépendants (r ≈ -0.83) → OK ensemble
>
> **Modifier la liste ci-dessous selon tes résultats.**


In [ ]:
# =============================================================================
# ★ DÉCISION POLLUANTS RETENUS — Modifier après avoir vu les heatmaps ★
# =============================================================================

CFG['polluants_retenus'] = ['PM25', 'O3']   # ← MODIFIER ICI selon heatmaps

# ── Mise à jour automatique de CFG ───────────────────────────────────────────
CFG['cumul'] = lca.detecter_vars_cumul(df_final, CFG)

CFG['tendance'] = [f'{pol}_mm365_sen_pente'
                   for pol in CFG['polluants_retenus']
                   if f'{pol}_mm365_sen_pente' in df_final.columns]

# Filtrer les seuils BH sur polluants_retenus uniquement
CFG['pct_pm25'] = seuils_retenus.get('PM25',[]) if 'PM25' in CFG['polluants_retenus'] else []
CFG['pct_pm10'] = seuils_retenus.get('PM10',[]) if 'PM10' in CFG['polluants_retenus'] else []
CFG['pct_o3']   = seuils_retenus.get('O3',  []) if 'O3'   in CFG['polluants_retenus'] else []

# Dose-réponse selon polluants_retenus
cumul_pm25 = [c for c in CFG['cumul'] if 'PM25' in c]
cumul_o3   = [c for c in CFG['cumul'] if 'O3'   in c]
CFG['polluant_dr_AB'] = cumul_pm25[0] if cumul_pm25 else CFG['cumul'][0] if CFG['cumul'] else None
CFG['polluant_dr_CD'] = cumul_o3[0]   if cumul_o3   else CFG['cumul'][0] if CFG['cumul'] else None

print(f"{'='*60}")
print(f"✅ CFG finalisé après décision heatmaps")
print(f"   polluants_retenus : {CFG['polluants_retenus']}")
print(f"   cumul             : {CFG['cumul']}")
print(f"   tendance          : {CFG['tendance']}")
print(f"   pct_pm25          : {CFG['pct_pm25']}")
print(f"   pct_pm10          : {CFG['pct_pm10']}")
print(f"   pct_o3            : {CFG['pct_o3']}")
print(f"   icpe              : {CFG['icpe']}")
print(f"{'='*60}")
print(f"\n→ Toutes les analyses suivantes utilisent ces paramètres.")


In [ ]:
# Heatmap 2 finale — variables retenues pour les modèles
pct_vars_h2 = ([f'PM25_pct_sup{s}' for s in CFG['pct_pm25']] +
               [f'PM10_pct_sup{s}' for s in CFG['pct_pm10']] +
               [f'O3_pct_sup{s}'   for s in CFG['pct_o3']])
vars_h2_final = {
    'Cumul (retenus)'    : CFG['cumul'],
    'Tendance (retenus)' : CFG['tendance'],
    'Clinique'           : ['age_diagnostic','paquet_annee','sexe_bin'],
    'Socio-éco'          : ['quintileEDI2021'],
    'Routier'            : ['indice_trafic'],
    'Industriel'         : CFG['icpe'],
}
if pct_vars_h2:
    vars_h2_final['% du temps (BH)'] = [v for v in pct_vars_h2 if v in df_final.columns]
lca.heatmap_correlation(df_final, vars_h2_final)


#### 📊 Résultats — Heatmap 2 finale — variables modèles

*À compléter après exécution.*

--- ## PARTIE 5d — Collinéarité ICPE + nettoyage automatique

> **Problème** : `nb_ICPE_total_3km` est la somme de `nb_ICPE_NS_3km + nb_ICPE_SB_3km + nb_ICPE_SH_3km`
> → corrélation ≈ 0.99 → **multicolinéarité** → OR aberrants dans les modèles (IC = [0.005 ; 258])
>
> Cette cellule :
> 1. Affiche la matrice de corrélation entre variables ICPE
> 2. Calcule les VIF (Variance Inflation Factor)
> 3. **Met à jour automatiquement `CFG['icpe']`** en retirant les variables redondantes
>
> ⚙️ Tu peux ajuster les seuils : `seuil_r=0.85` et `seuil_vif=5.0`

In [ ]:
# Analyse collinéarité ICPE → met à jour CFG['icpe'] automatiquement
icpe_nettoye = lca.analyser_collinearite_icpe(
    df_final,
    CFG,
    seuil_r   = 0.85,   # ← seuil corrélation : |r| > 0.85 = redondant
    seuil_vif = 5.0     # ← seuil VIF : > 5 = problématique
)

print(f"\nCFG['icpe'] avant → après nettoyage :")
print(f"  Avant : {lca.detecter_vars_icpe(df_final, {**CFG, 'icpe': lca.detecter_vars_icpe(df_final, CFG)})}")
print(f"  Après : {CFG['icpe']}")

#### 📊 Résultats — Collinéarité ICPE

*À compléter après exécution.*

Variables supprimées : *(ex: `nb_ICPE_total_3km` car r=0.99 avec `nb_ICPE_NS_3km`)*

Variables conservées dans `CFG['icpe']` : *liste après nettoyage*

---
## PARTIE 5c — Analyses monovariées

> Exécutée **après** la Partie 5b car utilise les variables finales :
> `polluants_retenus`, 1 seul seuil % par polluant retenu par BH, ICPE de CFG.
>
> Teste chaque variable séparément pour chaque groupe :
> - Variables continues → **Mann-Whitney** + OR brut (régression univariée)
> - Variables catégorielles → **Chi² / Fisher**


In [ ]:
# Analyses monovariées — variables finales uniquement
# (1 seul seuil % par polluant retenu par BH, ICPE de CFG)
print(f"Configuration utilisée :")
print(f"  polluants_retenus : {CFG['polluants_retenus']}")
print(f"  pct_pm25 retenu   : {CFG['pct_pm25']}")
print(f"  pct_o3 retenu     : {CFG['pct_o3']}")
print(f"  icpe              : {CFG['icpe']}")
print()
res_mono = lca.analyses_monovariees(df_final, CFG)


#### 📊 Résultats — Analyses monovariées (Partie 5c)

*À compléter après exécution.*

---
## PARTIE 6 — Tableaux de caractéristiques

In [ ]:
t1, t2, t3 = lca.tableaux_caracteristiques(df_final)

#### 📊 Résultats — Tableaux de caractéristiques

*À compléter après exécution.*

---
## PARTIE 7 — Régressions logistiques principales

> Variables définies dans CFG (seuils BH, polluants retenus après 5b).

In [ ]:
resultats = lca.analyses_principales(df_final, CFG)

#### 📊 Résultats — Régressions principales

*À compléter après exécution.*

---
## PARTIE 7b — Lasso sélection exposition

> Les covariables cliniques (tabac, sexe, âge, EDI, trafic) sont **forcées** dans le modèle.
> Le Lasso pénalise uniquement les **variables d'exposition** (polluants, ICPE).
>
> **Question :** parmi les variables d'exposition disponibles, lesquelles apportent
> une information discriminante au-delà des facteurs cliniques ?


In [ ]:
# Lasso exposition — covariables cliniques forcées, exposition pénalisée
res_lasso_expo_AB   = lca.lasso_selection_exposition(df_final, CFG, groupe='AB')
res_lasso_expo_CD   = lca.lasso_selection_exposition(df_final, CFG, groupe='CD')
res_lasso_expo_ACBD = lca.lasso_selection_exposition(df_final, CFG, groupe='ACBD')

print("\n── Synthèse Lasso exposition ──")
for lbl, res in [('A vs B', res_lasso_expo_AB),
                  ('C vs D', res_lasso_expo_CD),
                  ('A+C vs B+D', res_lasso_expo_ACBD)]:
    if res:
        print(f"  {lbl} : AUC CV={res['auc_cv']:.3f} | "
              f"Variables : {res['vars_selectionnees'] or ['aucune']}")


#### 📊 Résultats — Lasso sélection exposition (Partie 7b)

*À compléter après exécution.*

---
## PARTIE 8 — Mutations individuelles

| Mutation | N+  | EPV  | Méthode |
|---|---|---|---|
| EGFR  | 178 | 16.2 | Logistique std ✅ |
| MET   | 58  | 5.3  | **Firth** ⚠️ |
| ALK   | 54  | 4.9  | **Firth** ⚠️ |
| ERBB2 | 32  | 2.9  | **Firth** ❌ |
| ROS1  | 30  | 2.7  | **Firth** ❌ |


In [ ]:
resultats_mutations = lca.analyses_mutations(df_final, CFG, mutations=['EGFR','MET','ALK','ERBB2','ROS1'])

#### 📊 Résultats — Mutations individuelles

*À compléter après exécution.*

---
## PARTIE 9 — Analyses de sensibilité

In [ ]:
res_sexe_AB   = lca.analyse_stratifiee_sexe(df_final, CFG, groupe='AB')
res_sexe_CD   = lca.analyse_stratifiee_sexe(df_final, CFG, groupe='CD')


#### 📊 Résultats — Stratification sexe

*À compléter après exécution.*

In [ ]:
res_nonfumeurs = lca.analyse_sous_groupe(df_final, CFG,
    filtre_fumeur=False, groupe='AB', titre_custom='A vs B — Non-fumeurs uniquement')
res_adeno = lca.analyse_sous_groupe(df_final, CFG,
    filtre_histologie='adenocarcinome', groupe='AB', titre_custom='A vs B — Adénocarcinomes')


#### 📊 Résultats — Sous-groupes

*À compléter après exécution.*

--- ## PARTIE 9b — Analyse de sensibilité : ajustement pour le radon

> **Question de l'enseignant** : L'effet des polluants atmosphériques persiste-t-il après
> ajustement pour le radon ?
>
> Compare les OR des polluants dans deux modèles :
> - **Modèle sans radon** : polluants + covariables (comme Partie 7)
> - **Modèle avec radon** : polluants + covariables + radon_categorie
>
> Si |ΔOR| < 10% → effet pollution **robuste**. Le radon est un confondant faible.
> Si |ΔOR| > 20% → le radon est un **confondant fort**.

In [ ]:
res_sensibilite_radon = lca.analyse_sensibilite_radon(df_final, CFG)

# ── Analyse dose-réponse radon (quartiles) ──────────────────────────────────
# Teste si le risque croît avec le score radon (Q1 → Q4)
# Utilise radon_quartile créé automatiquement par ajouter_radon()
if 'radon_quartile' in df_final.columns:
    for groupe in ['AB', 'CD', 'ACBD']:
        lca.dose_reponse_quintiles(
            df_final, CFG,
            polluant='radon_score',   # variable continue
            groupe=groupe,
            n_quintiles=4             # 4 quartiles
        )

# Note : si CFG['inclure_radon'] = True, les modèles principaux (Partie 7)
# incluent déjà radon_score. Cette cellule compare systématiquement
# avec vs sans radon, indépendamment de CFG['inclure_radon'].

#### 📊 Résultats — Sensibilité au radon

*À compléter après exécution.*

Pour chaque groupe :
- **AUC sans radon** vs **AUC avec radon** → gain d'AUC dû au radon
- **OR radon** → OR propre du radon (indépendamment des polluants)
- **ΔOR (%)** → % de changement de l'OR des polluants après ajout du radon

---
## PARTIE 10 — Analyses dose-réponse

In [ ]:
dr_AB = lca.dose_reponse_quintiles(df_final, CFG,
    polluant=CFG['polluant_dr_AB'], groupe='AB', n_quintiles=CFG['n_quintiles'])
dr_CD = lca.dose_reponse_quintiles(df_final, CFG,
    polluant=CFG['polluant_dr_CD'], groupe='CD', n_quintiles=CFG['n_quintiles'])
ds_egfr = lca.dose_reponse_spatiale(df_final, CFG,
    col_distance=CFG['col_distance'], mutation='EGFR', tranches_km=CFG['tranches_km'])
ds_AB = lca.dose_reponse_spatiale(df_final, CFG,
    col_distance=CFG['col_distance'], groupe='AB', tranches_km=CFG['tranches_km'])


#### 📊 Résultats — Dose-réponse

*À compléter après exécution.*

---
## PARTIE 11 — Variables % du temps

In [ ]:
df_res_pct = lca.analyse_pct_temps(df_final, CFG)

#### 📊 Résultats — Variables % du temps

*À compléter après exécution.*

---
## PARTIE 11b — Sensibilité multi-fenêtres × polluants

In [ ]:
combinaisons = [
    {'fenetre_ans':3, 'polluants_retenus':['PM25','O3'], 'label':'10ans_PM25_O3'},
    {'fenetre_ans':5,  'polluants_retenus':['PM25','O3'], 'label':'3ans_PM25_O3'},
    # {'fenetre_ans':10, 'polluants_retenus':['PM10','O3'], 'label':'10ans_PM10_O3'},
]  # ← modifier librement

import pandas as pd; rows_f = []
for combo in combinaisons:
    lbl = combo.pop('label')
    print(f"\n{'█'*40}\n{lbl}")
    cfg_c = {**CFG, **combo}
    df_air_c = lca.calculer_variables_air(data, df_clinique, cfg_c)
    df_air_c = lca.ajouter_socioeco(df_air_c, df_clinique, CHEMINS['edi'])
    df_air_c, _ = lca.ajouter_routier(df_air_c, df_clinique, cfg_c)
    df_air_c = lca.ajouter_icpe(df_air_c, gdf_patients, cfg_c)
    df_c = lca.construire_df_final(data, df_clinique, df_air_c)
    cfg_c['cumul']   = lca.detecter_vars_cumul(df_c, cfg_c)
    cfg_c['tendance']= [f'{p}_mm365_sen_pente' for p in cfg_c['polluants_retenus']
                        if f'{p}_mm365_sen_pente' in df_c.columns]
    _, s_f = lca.chercher_seuils_critiques(df_c, {**cfg_c,'polluants_retenus':cfg_c['polluants_retenus']})
    cfg_c['pct_pm25'] = s_f.get('PM25',[]) or ([10] if 'PM25' in cfg_c['polluants_retenus'] else [])
    cfg_c['pct_pm10'] = s_f.get('PM10',[]) or ([35] if 'PM10' in cfg_c['polluants_retenus'] else [])
    cfg_c['pct_o3']   = s_f.get('O3',  []) or ([100] if 'O3'  in cfg_c['polluants_retenus'] else [])
    cfg_c['icpe'] = lca.detecter_vars_icpe(df_c, cfg_c)
    res_c = lca.analyses_principales(df_c, cfg_c)
    for g,r in res_c.items():
        if r and r.get('auc'):
            rows_f.append({'Combinaison':lbl,'Groupe':g,'AUC':round(r['auc'],3)})
df_fenetres = pd.DataFrame(rows_f).pivot(index='Groupe',columns='Combinaison',values='AUC')
print("\n── AUC par combinaison ──")
print(df_fenetres.to_string())


#### 📊 Résultats — Multi-fenêtres × polluants

*À compléter après exécution.*

---
## PARTIE 11e — Seuil concentration optimal C* (données journalières brutes)

> Pour chaque polluant retenu, trouve la concentration journalière C*
> qui discrimine le mieux les groupes, via courbe ROC sur la médiane
> journalière par patient (indépendance des observations).


In [ ]:
resultats_c_star = lca.trouver_seuil_concentration_optimal(
    data, df_clinique, df_final, CFG)


#### 📊 Résultats — C* sur données brutes

*À compléter après exécution.*

---
## PARTIE 11f — % du temps au-dessus de C* → Seuil P*

> Calcule pour chaque patient le % de jours au-dessus de C* (trouvé en 11e).
> Puis cherche le seuil de % optimal P* par Youden.


In [ ]:
df_final, res_p_star = lca.calculer_pct_seuil_optimal(
    data, df_clinique, df_final, CFG, resultats_c_star)
print(f"\n✅ df_final enrichi : {len(df_final.columns)} variables")


#### 📊 Résultats — % du temps au-dessus de C* + P*

*À compléter après exécution.*

---
## PARTIE 11g — Seuils Youden sur cumul et pente

> Trouve le seuil optimal par Youden sur `PM25_cumul_120m` et `PM25_mm365_sen_pente`.
> Crée les variables binaires correspondantes dans df_final.


In [ ]:
df_final, seuils_youden_continus = lca.trouver_seuils_youden_continus(
    df_final, CFG)
print(f"\n✅ df_final enrichi : {len(df_final.columns)} variables")


#### 📊 Résultats — Seuils Youden cumul et pente

*À compléter après exécution.*

---
## PARTIE 11c — Seuils Youden sur variables % arbitraires (df_final)

> Compare les seuils Youden trouvés sur les variables déjà dans df_final
> (PM25_pct_sup5, PM25_pct_sup10...) avec les seuils BH de la Partie 4.


In [ ]:
res_seuils_bh = {
    'df_res': CFG.get('_df_res_seuils', pd.DataFrame()),
}
res_seuils_opt = lca.chercher_seuils_optimaux(
    df_final, CFG, comparer_partie4=True)


#### 📊 Résultats — Seuils Youden variables arbitraires

*À compléter après exécution.*

---
## PARTIE 11h — Synthèse comparative : manuelle vs automatique

> Compare les AUC univariées obtenues avec :
> - Approche manuelle : seuils arbitraires BH
> - Approche automatique : C* (données brutes) + Youden


In [ ]:
df_synthese_comp = lca.synthese_approches(
    df_final       = df_final,
    cfg            = CFG,
    res_seuils_bh  = {'df_res': CFG.get('_df_res_seuils', pd.DataFrame())},
    resultats_c_star = resultats_c_star,
    res_p_star       = res_p_star,
    seuils_youden_continus = seuils_youden_continus,
)


#### 📊 Résultats — Synthèse comparative manuelle vs automatique

*À compléter après exécution.*

---
## PARTIE 11i — Régressions finales comparatives

> **Série A** : variables manuelles BH (identique Partie 7)
> **Série B** : variables automatiques Youden (C*, P*, binaires cumul/pente)
> → Comparer AUC et OR significatifs


In [ ]:
res_comp = lca.regressions_comparatives(
    df_final              = df_final,
    cfg                   = CFG,
    res_p_star            = res_p_star,
    seuils_youden_continus= seuils_youden_continus,
)

cfg_youden_final = res_comp['cfg_youden']
print(f"\nSérie B — Variables Youden utilisées :")
print(f"  icpe (+ Youden) : {cfg_youden_final['icpe']}")


#### 📊 Résultats — Régressions comparatives A vs B

*À compléter après exécution.*

---
## PARTIE 12 — Synthèse générale

In [ ]:
import pandas as pd
df_synthese = lca.synthese_comparaison({
    'A vs B — BH (Partie 7)'       : resultats.get('AB'),
    'C vs D — BH (Partie 7)'       : resultats.get('CD'),
    'A+C vs B+D — BH (Partie 7)'   : resultats.get('ACBD'),
    'EGFR ✅'                       : resultats_mutations.get('EGFR'),
    'A vs B — Youden (Partie 11i)' : res_comp['serie_B'].get('AB'),
    'C vs D — Youden (Partie 11i)' : res_comp['serie_B'].get('CD'),
    'Non-fumeurs'                  : res_nonfumeurs,
    'Adénocarcinomes'              : res_adeno,
    'A vs B — Femmes'              : res_sexe_AB.get('feminin'),
    'A vs B — Hommes'              : res_sexe_AB.get('masculin'),
})


## ANALYSE GROUPE MUTATION EGFR ALK MET VS PATIENTS NON FUMEURS NON MUTES

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  ANALYSE CUSTOM — EGFR + MET + ALK  vs  Non-mutés Non-fumeurs (3 ans)  ║
# ╚══════════════════════════════════════════════════════════════════════════╝
#
# Groupe 1 : patients porteurs de EGFR OU MET OU ALK
# Groupe 2 : patients NON-fumeurs (paquet_annee == 0)
#            ET sans aucune mutation NF (ni EGFR, ALK, RET, MET, ERBB2, ROS1, NTRK)
#
# Intérêt : comparaison "pure" — mutations actionnables vs cas inexpliqués
# → si la pollution joue un rôle, elle devrait différencier ces deux groupes

import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
import matplotlib.pyplot as plt

# ── 1. Définition des groupes ─────────────────────────────────────────────────

MUTATIONS_CIBLE = ['EGFR', 'MET', 'ALK']

# Groupe 1 — porteurs EGFR ou MET ou ALK
mask_g1 = pd.Series(False, index=df_final.index)
for mut in MUTATIONS_CIBLE:
    mask_g1 |= lca.porte_mutation(df_final, mut)

# Groupe 2 — non-fumeurs sans aucune mutation NF
mask_g2 = (
    (df_final['paquet_annee'] == 0)
    & (df_final['mutation'].str.upper() == 'NO MUTATION')
)

# Vérifier que les deux groupes sont disjoints
overlap = (mask_g1 & mask_g2).sum()
print(f"Overlap G1∩G2 : {overlap} patients")   # doit être 0 ou quasi-0
# (un patient EGFR/MET/ALK est en G1 même s'il est non-fumeur — cf. note ci-dessous)
# Note : certains patients G1 peuvent être non-fumeurs → ils restent en G1, pas en G2

# Construction du DataFrame de travail
mask_combined = mask_g1 | mask_g2
df_c = df_final[mask_combined].copy()
df_c['outcome'] = mask_g1[mask_combined].astype(int)   # 1 = G1, 0 = G2

n_g1    = int(df_c['outcome'].sum())
n_g2    = int((df_c['outcome'] == 0).sum())
n_total = len(df_c)
n_vars  = len(lca._vars_modele(CFG, inclure_paquet=False))   # sans paquet (G2 = tous 0)
epv     = n_g1 / max(1, n_vars)

print(f"\n{'═'*58}")
print(f"  EGFR+MET+ALK  vs  Non-mutés Non-fumeurs — 3 ans")
print(f"{'═'*58}")
print(f"  Groupe 1 (EGFR+MET+ALK)      : n = {n_g1}")
print(f"  Groupe 2 (Non-mutés NF)       : n = {n_g2}")
print(f"  Total                         : N = {n_total}")
print(f"  Variables modèle              : {n_vars}")
print(f"  EPV                           : {epv:.1f}  {'✅' if epv >= 10 else ('⚠️' if epv >= 5 else '❌')}")
print(f"  ⚠️  paquet_annee EXCLU du modèle")
print(f"     (G2 = tous non-fumeurs → colinéarité parfaite avec l'outcome)")

# ── 2. Variables du modèle ────────────────────────────────────────────────────
#
# paquet_annee est EXCLU : par construction, G2 = paquet_annee==0
# → ajouter paquet_annee dans ce modèle créerait une colinéarité quasi-parfaite
# On conserve : polluants (cumul + tendance + pct) + âge + sexe + EDI + trafic + radon + IPG + ICPE

vars_mod = lca._vars_modele(CFG, inclure_paquet=False)   # sans tabac
vars_mod = [v for v in vars_mod if v in df_c.columns]

df_reg = df_c[vars_mod + ['outcome']].dropna()
n_reg  = len(df_reg)
print(f"\n  N après dropna              : {n_reg}")
print(f"  Variables retenues ({len(vars_mod)})      : {vars_mod}")

if n_reg < 30 or df_reg['outcome'].sum() < 5:
    print("⛔ Effectif insuffisant pour la régression.")
    raise StopIteration

# ── 3. Standardisation + régression logistique ───────────────────────────────

scaler   = StandardScaler()
X_scaled = scaler.fit_transform(df_reg[vars_mod])
df_std   = pd.DataFrame(X_scaled, columns=vars_mod, index=df_reg.index)
df_std['outcome'] = df_reg['outcome'].values

vars_ok  = [v for v in vars_mod if float(df_std[v].std()) > 1e-8]
X        = sm.add_constant(df_std[vars_ok])
y        = df_std['outcome']

# Firth si EPV < 10
if epv < 10:
    try:
        from statsmodels.discrete.discrete_model import Logit as FirthLogit
        import statsmodels.formula.api as smf
        # Utilise la régression de Firth via logistf si disponible
        mod = sm.Logit(y, X).fit_regularized(method='l1', alpha=0.1, disp=False)
        methode = "Logit régularisé (L1)"
    except Exception:
        mod = sm.Logit(y, X).fit(disp=False)
        methode = "Logit standard (EPV faible ⚠️)"
else:
    mod = sm.Logit(y, X).fit(disp=False)
    methode = "Logit standard"

auc = roc_auc_score(y, mod.predict(X))
print(f"\n  Méthode : {methode}")
print(f"  AUC     : {auc:.3f}")
print(f"  R²McFadden : {mod.prsquared:.3f}")
print(f"  AIC     : {mod.aic:.1f}")

# ── 4. Tableau des OR ─────────────────────────────────────────────────────────

tab_or = lca._tableau_OR(mod, vars_ok)

print(f"\n  {'─'*58}")
print(f"  RÉSULTATS — OR ajustés (pour 1 ET exposé/SD)")
print(f"  Outcome : 1 = EGFR+MET+ALK | 0 = Non-mutés Non-fumeurs")
print(f"  {'─'*58}")
try:
    from IPython.display import display
    display(tab_or.style
        .background_gradient(subset=['OR'], cmap='RdYlGn', vmin=0.5, vmax=2.0)
        .format({'OR': '{:.3f}', 'IC 95% inf': '{:.3f}', 'IC 95% sup': '{:.3f}',
                 'p-value': lambda x: '<0.001' if x == '<0.001' else f'{float(x):.3f}'
                             if x not in ['<0.001'] else x}))
except Exception:
    print(tab_or.to_string(index=False))

# ── 5. Forest plot — couleurs uniquement sur les variables significatives ──

fig, ax = plt.subplots(figsize=(10, max(5, len(vars_ok) * 0.55 + 1.5)))
ax.axvline(x=1, color='black', linestyle='--', lw=1.2, alpha=0.7)

COULEURS = {
    'exposition': '#2E86AB',
    'radon':      '#E76F51',
    'ipg':        '#52B788',
    'clinique':   '#2E86AB',   # même couleur que exposition si sig
}
GRIS = "#000000"

def _categorie_var(v):
    if 'radon' in v:                                              return 'radon'
    if v == 'IPG':                                                return 'ipg'
    if any(p in v for p in ['PM25','PM10','O3','NO2',
                             'ICPE','dist_','nb_']):              return 'exposition'
    return 'clinique'

# Variables significatives
sig_set = set(tab_or[(tab_or['Variable'] != 'const') &
                     (tab_or['p_num'] < 0.05)]['Variable'].tolist())

tab_plot = tab_or[tab_or['Variable'] != 'const'].copy().reset_index(drop=True)

for i, row in tab_plot.iterrows():
    var  = row['Variable']
    or_  = row['OR']
    lo   = row['IC 95% inf']
    hi   = row['IC 95% sup']
    sig  = var in sig_set

    if sig:
        col        = COULEURS[_categorie_var(var)]
        lw         = 2.2
        markersize = 9
        alpha      = 1.0
        label_kw   = dict(fontsize=8.5, fontweight='bold', color=col)
    else:
        col        = GRIS
        lw         = 1.2
        markersize = 6
        alpha      = 0.6
        label_kw   = dict(fontsize=8, fontweight='normal', color="#000000")

    ax.plot([lo, hi], [i, i], color=col, lw=lw, alpha=alpha)
    ax.plot(or_, i, 'o', color=col, markersize=markersize, alpha=alpha,
            markeredgecolor='white', markeredgewidth=0.8)

    # Annoter uniquement les significatives
    if sig:
        ax.text(max(hi * 1.04, 1.05), i,
                f"OR={or_:.2f}  p={row['p-value']}", va='center', **label_kw)
    else:
        ax.text(max(hi * 1.04, 1.05), i,
                f"OR={or_:.2f}", va='center', **label_kw)

# Labels axe Y : gras si significatif, gris sinon
ytick_labels = []
for var in tab_plot['Variable']:
    if var in sig_set:
        ytick_labels.append(var)
    else:
        ytick_labels.append(var)

ax.set_yticks(range(len(tab_plot)))
ax.set_yticklabels(tab_plot['Variable'].tolist(), fontsize=9)

# Colorer les ytick labels aussi
for tick, var in zip(ax.get_yticklabels(), tab_plot['Variable']):
    if var in sig_set:
        tick.set_color(COULEURS[_categorie_var(var)])
        tick.set_fontweight('bold')
    else:
        tick.set_color("#000000")

ax.set_xscale('log')
ax.set_xlabel('Odds Ratio (IC 95%)  — échelle log', fontsize=10)
ax.set_title(
    f'EGFR+MET+ALK  vs  NON MUTATED AND NEVER SMOKER  |  3 years\n'
    f'N={n_reg} | AUC={auc:.3f} | R²McF={mod.prsquared:.3f} | '
    f'{len(sig_set)} Significatives variables',
    fontweight='bold', fontsize=11
)

from matplotlib.lines import Line2D
legende = [
    Line2D([0],[0], color=COULEURS['exposition'], lw=3, label='Pollutant / ICPE (sig.)'),
    Line2D([0],[0], color=COULEURS['ipg'],        lw=3, label='IPG (sig.)'),
    Line2D([0],[0], color=COULEURS['radon'],      lw=3, label='Radon (sig.)'),
    Line2D([0],[0], color=GRIS,                   lw=2, alpha=0.6, label='Non significative'),
]
ax.legend(handles=legende, fontsize=9, loc='lower right')
ax.grid(True, axis='x', alpha=0.3, linestyle=':')
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.show()

# Légende couleurs
from matplotlib.lines import Line2D
legende = [
    Line2D([0],[0], color=COULEURS['exposition'], lw=3, label='Pollutants / ICPE'),
    Line2D([0],[0], color=COULEURS['clinique'],   lw=3, label='Clinical variables'),
    Line2D([0],[0], color=COULEURS['radon'],      lw=3, label='Radon'),
    Line2D([0],[0], color=COULEURS['ipg'],        lw=3, label='IPG'),
]
ax.legend(handles=legende, fontsize=9, loc='lower right')
ax.grid(True, axis='x', alpha=0.3, linestyle=':')
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.show()

# ── 6. Résumé synthétique ─────────────────────────────────────────────────────

# ── Correction synthèse — utilise p_num au lieu de Sig ──
sig_vars = tab_or[
    (tab_or['Variable'] != 'const') & 
    (tab_or['p_num'] < 0.05)
]['Variable'].tolist()

print(f"\n  ── Synthèse corrigée ──")
print(f"  Variables significatives (p<0.05) : {sig_vars if sig_vars else 'aucune'}")

for _, row in tab_or[tab_or['Variable'].isin(sig_vars)].iterrows():
    direction = f"→ associé à {'EGFR+MET+ALK' if row['OR'] > 1 else 'Non-mutés NF'}"
    print(f"    {row['Variable']:35s}  OR={row['OR']:.3f}  p={row['p-value']}  {direction}")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  ANALYSE CUSTOM — EGFR + MET + ALK  vs  Non-mutés Non-fumeurs (3 ans)  ║
# ╚══════════════════════════════════════════════════════════════════════════╝
#
# Groupe 1 : patients porteurs de EGFR OU MET OU ALK
# Groupe 2 : patients NON-fumeurs (paquet_annee == 0)
#            ET sans aucune mutation NF (ni EGFR, ALK, RET, MET, ERBB2, ROS1, NTRK)
#
# Intérêt : comparaison "pure" — mutations actionnables vs cas inexpliqués
# → si la pollution joue un rôle, elle devrait différencier ces deux groupes

import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
import matplotlib.pyplot as plt

# ── 1. Définition des groupes ─────────────────────────────────────────────────

MUTATIONS_CIBLE = ['EGFR', 'MET', 'ALK']

for MUTATION_CIBLE in ['EGFR', 'ALK', 'MET']:

    print("\n" + "═"*70)
    print(f"ANALYSE : {MUTATION_CIBLE} vs Non-mutés Non-fumeurs")
    print("═"*70)

    # Groupe 1 — porteurs EGFR ou MET ou ALK
    mask_g1 = lca.porte_mutation(df_final, MUTATION_CIBLE)

    # Groupe 2 — non-fumeurs sans aucune mutation NF
    mask_g2 = (
        (df_final['paquet_annee'] == 0)
        & (df_final['mutation'].str.upper() == 'NO MUTATION')
    )

    # Vérifier que les deux groupes sont disjoints
    overlap = (mask_g1 & mask_g2).sum()
    print(f"Overlap G1∩G2 : {overlap} patients")   # doit être 0 ou quasi-0
    # (un patient EGFR/MET/ALK est en G1 même s'il est non-fumeur — cf. note ci-dessous)
    # Note : certains patients G1 peuvent être non-fumeurs → ils restent en G1, pas en G2

    # Construction du DataFrame de travail
    mask_combined = mask_g1 | mask_g2
    df_c = df_final[mask_combined].copy()
    df_c['outcome'] = mask_g1[mask_combined].astype(int)   # 1 = G1, 0 = G2

    n_g1    = int((df_c['outcome'] == 1).sum())
    if n_g1 < 20:
        print(f"⚠️ {MUTATION_CIBLE} : effectif trop faible ({n_g1})")
        continue
    n_g2    = int((df_c['outcome'] == 0).sum())
    n_total = len(df_c)
    n_vars  = len(lca._vars_modele(CFG, inclure_paquet=False))   # sans paquet (G2 = tous 0)
    epv     = n_g1 / max(1, n_vars)

    print(f"\n{'═'*58}")
    print(f"  {MUTATION_CIBLE}  vs  Non-mutés Non-fumeurs — 3 ans")
    print(f"{'═'*58}")
    print(f"  {MUTATION_CIBLE}               : n = {n_g1}")
    print(f"  Non-mutés NF        : n = {n_g2}")
    print(f"  Total                         : N = {n_total}")
    print(f"  Variables modèle              : {n_vars}")
    print(f"  EPV                           : {epv:.1f}  {'✅' if epv >= 10 else ('⚠️' if epv >= 5 else '❌')}")
    print(f"  ⚠️  paquet_annee EXCLU du modèle")
    print(f"     (G2 = tous non-fumeurs → colinéarité parfaite avec l'outcome)")

    # ── 2. Variables du modèle ────────────────────────────────────────────────────
    #
    # paquet_annee est EXCLU : par construction, G2 = paquet_annee==0
    # → ajouter paquet_annee dans ce modèle créerait une colinéarité quasi-parfaite
    # On conserve : polluants (cumul + tendance + pct) + âge + sexe + EDI + trafic + radon + IPG + ICPE

    vars_mod = lca._vars_modele(CFG, inclure_paquet=False)   # sans tabac
    vars_mod = [v for v in vars_mod if v in df_c.columns]

    df_reg = df_c[vars_mod + ['outcome']].dropna()
    n_reg  = len(df_reg)
    print(f"\n  N après dropna              : {n_reg}")
    print(f"  Variables retenues ({len(vars_mod)})      : {vars_mod}")

    if n_reg < 30 or df_reg['outcome'].sum() < 5:
        print("⛔ Effectif insuffisant pour la régression.")
        raise StopIteration

    # ── 3. Standardisation + régression logistique ───────────────────────────────

    scaler   = StandardScaler()
    X_scaled = scaler.fit_transform(df_reg[vars_mod])
    df_std   = pd.DataFrame(X_scaled, columns=vars_mod, index=df_reg.index)
    df_std['outcome'] = df_reg['outcome'].values

    vars_ok  = [v for v in vars_mod if float(df_std[v].std()) > 1e-8]
    X        = sm.add_constant(df_std[vars_ok])
    y        = df_std['outcome']

    # Firth si EPV < 10
    if epv < 10:
        try:
            from statsmodels.discrete.discrete_model import Logit as FirthLogit
            import statsmodels.formula.api as smf
            # Utilise la régression de Firth via logistf si disponible
            mod = sm.Logit(y, X).fit_regularized(method='l1', alpha=0.1, disp=False)
            methode = "Logit régularisé (L1)"
        except Exception:
            mod = sm.Logit(y, X).fit(disp=False)
            methode = "Logit standard (EPV faible ⚠️)"
    else:
        mod = sm.Logit(y, X).fit(disp=False)
        methode = "Logit standard"

    auc = roc_auc_score(y, mod.predict(X))
    print(f"\n  Méthode : {methode}")
    print(f"  AUC     : {auc:.3f}")
    print(f"  R²McFadden : {mod.prsquared:.3f}")
    print(f"  AIC     : {mod.aic:.1f}")

    # ── 4. Tableau des OR ─────────────────────────────────────────────────────────

    tab_or = lca._tableau_OR(mod, vars_ok)

    print(f"\n  {'─'*58}")
    print(f"  RÉSULTATS — OR ajustés (pour 1 ET exposé/SD)")
    print(f"  Outcome : 1 = {MUTATION_CIBLE} | 0 = Non-mutés Non-fumeurs")
    print(f"  {'─'*58}")
    try:
        from IPython.display import display
        display(tab_or.style
            .background_gradient(subset=['OR'], cmap='RdYlGn', vmin=0.5, vmax=2.0)
            .format({'OR': '{:.3f}', 'IC 95% inf': '{:.3f}', 'IC 95% sup': '{:.3f}',
                    'p-value': lambda x: '<0.001' if x == '<0.001' else f'{float(x):.3f}'
                                if x not in ['<0.001'] else x}))
    except Exception:
        print(tab_or.to_string(index=False))

    # ── 5. Forest plot — couleurs uniquement sur les variables significatives ──

    fig, ax = plt.subplots(figsize=(10, max(5, len(vars_ok) * 0.55 + 1.5)))
    ax.axvline(x=1, color='black', linestyle='--', lw=1.2, alpha=0.7)

    COULEURS = {
        'exposition': '#2E86AB',
        'radon':      '#E76F51',
        'ipg':        '#52B788',
        'clinique':   '#2E86AB',   # même couleur que exposition si sig
    }
    GRIS = "#000000"

    def _categorie_var(v):
        if 'radon' in v:                                              return 'radon'
        if v == 'IPG':                                                return 'ipg'
        if any(p in v for p in ['PM25','PM10','O3','NO2',
                                'ICPE','dist_','nb_']):              return 'exposition'
        return 'clinique'

    # Variables significatives
    sig_set = set(tab_or[(tab_or['Variable'] != 'const') &
                        (tab_or['p_num'] < 0.05)]['Variable'].tolist())

    tab_plot = tab_or[tab_or['Variable'] != 'const'].copy().reset_index(drop=True)

    for i, row in tab_plot.iterrows():
        var  = row['Variable']
        or_  = row['OR']
        lo   = row['IC 95% inf']
        hi   = row['IC 95% sup']
        sig  = var in sig_set

        if sig:
            col        = COULEURS[_categorie_var(var)]
            lw         = 2.2
            markersize = 9
            alpha      = 1.0
            label_kw   = dict(fontsize=8.5, fontweight='bold', color=col)
        else:
            col        = GRIS
            lw         = 1.2
            markersize = 6
            alpha      = 0.6
            label_kw   = dict(fontsize=8, fontweight='normal', color="#000000")

        ax.plot([lo, hi], [i, i], color=col, lw=lw, alpha=alpha)
        ax.plot(or_, i, 'o', color=col, markersize=markersize, alpha=alpha,
                markeredgecolor='white', markeredgewidth=0.8)

        # Annoter uniquement les significatives
        if sig:
            ax.text(max(hi * 1.04, 1.05), i,
                    f"OR={or_:.2f}  p={row['p-value']}", va='center', **label_kw)
        else:
            ax.text(max(hi * 1.04, 1.05), i,
                    f"OR={or_:.2f}", va='center', **label_kw)

    # Labels axe Y : gras si significatif, gris sinon
    ytick_labels = []
    for var in tab_plot['Variable']:
        if var in sig_set:
            ytick_labels.append(var)
        else:
            ytick_labels.append(var)

    ax.set_yticks(range(len(tab_plot)))
    ax.set_yticklabels(tab_plot['Variable'].tolist(), fontsize=9)

    # Colorer les ytick labels aussi
    for tick, var in zip(ax.get_yticklabels(), tab_plot['Variable']):
        if var in sig_set:
            tick.set_color(COULEURS[_categorie_var(var)])
            tick.set_fontweight('bold')
        else:
            tick.set_color("#000000")

    ax.set_xscale('log')
    ax.set_xlabel('Odds Ratio (IC 95%)  — échelle log', fontsize=10)
    ax.set_title(
    f'{MUTATION_CIBLE} vs NON-MUTATED AND  NEVER SMOKER | 3 years\n'
        f'N={n_reg} | AUC={auc:.3f} | R²McF={mod.prsquared:.3f} | '
        f'{len(sig_set)} significatives variables',
        fontweight='bold', fontsize=11
    )

    # from matplotlib.lines import Line2D
    # legende = [
    #     Line2D([0],[0], color=COULEURS['exposition'], lw=3, label='Polluant / ICPE (sig.)'),
    #     Line2D([0],[0], color=COULEURS['ipg'],        lw=3, label='IPG (sig.)'),
    #     Line2D([0],[0], color=COULEURS['radon'],      lw=3, label='Radon (sig.)'),
    #     Line2D([0],[0], color=GRIS,                   lw=2, alpha=0.6, label='Non significatif'),
    # ]
    # ax.legend(handles=legende, fontsize=9, loc='lower right')
    ax.grid(True, axis='x', alpha=0.3, linestyle=':')
    ax.spines[['top','right']].set_visible(False)
    plt.tight_layout()
    plt.show()

    # # Légende couleurs
    # from matplotlib.lines import Line2D
    # legende = [
    #     Line2D([0],[0], color=COULEURS['exposition'], lw=3, label='Polluants / ICPE'),
    #     Line2D([0],[0], color=COULEURS['clinique'],   lw=3, label='Variables cliniques'),
    #     Line2D([0],[0], color=COULEURS['radon'],      lw=3, label='Radon'),
    #     Line2D([0],[0], color=COULEURS['ipg'],        lw=3, label='IPG'),
    # ]
    # ax.legend(handles=legende, fontsize=9, loc='lower right')
    ax.grid(True, axis='x', alpha=0.3, linestyle=':')
    ax.spines[['top','right']].set_visible(False)
    plt.tight_layout()
    plt.show()

    # ── 6. Résumé synthétique ─────────────────────────────────────────────────────

    # ── Correction synthèse — utilise p_num au lieu de Sig ──
    sig_vars = tab_or[
        (tab_or['Variable'] != 'const') & 
        (tab_or['p_num'] < 0.05)
    ]['Variable'].tolist()

    print(f"\n  ── Synthèse corrigée ──")
    print(f"  Variables significatives (p<0.05) : {sig_vars if sig_vars else 'aucune'}")

    for _, row in tab_or[tab_or['Variable'].isin(sig_vars)].iterrows():
        direction = (
    f"→ associé à {MUTATION_CIBLE}"
    if row['OR'] > 1
    else "→ associé à Non-mutés NF"
)
        print(f"    {row['Variable']:35s}  OR={row['OR']:.3f}  p={row['p-value']}  {direction}")

In [ ]:
# Modèle A — O3_cumul sans PM25_cumul
vars_test_O3  = [v for v in vars_ok if 'PM25_cumul' not in v]

# Modèle B — PM25_cumul sans O3_cumul  
vars_test_PM25 = [v for v in vars_ok if 'O3_cumul' not in v]

for label, vars_test in [('O3 seul', vars_test_O3), ('PM25 seul', vars_test_PM25)]:
    X_t = sm.add_constant(df_std[vars_test])
    mod_t = sm.Logit(y, X_t).fit(disp=False)
    auc_t = roc_auc_score(y, mod_t.predict(X_t))
    tab_t = lca._tableau_OR(mod_t, vars_test)
    or_t  = tab_t.set_index('Variable')
    cumul_var = 'O3_cumul_36m' if 'O3 seul' in label else 'PM25_cumul_36m'
    if cumul_var in or_t.index:
        row = or_t.loc[cumul_var]
        print(f"\n{label}: OR={row['OR']:.3f} IC[{row['IC 95% inf']:.2f}–{row['IC 95% sup']:.2f}] p={row['p-value']} | AUC={auc_t:.3f}")